In [ ]:
# Standard Library Imports
import copy
import os
import time
from getpass import getpass
from uuid import uuid4

# Third-Party Imports
# LangChain Core
from langchain.prompts import ChatPromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

# LangChain Community
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

# LangChain Integrations
from langchain_cohere import CohereRerank
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

# LangGraph
from langgraph.graph import START, StateGraph

# Qdrant
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

# RAGAS
from ragas import EvaluationDataset, RunConfig, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    ContextEntityRecall,
    Faithfulness,
    FactualCorrectness,
    LLMContextRecall,
    NoiseSensitivity,
    ResponseRelevancy,
)
from ragas.testset import TestsetGenerator

# Typing
from typing_extensions import List, TypedDict

# Local Application Imports
# (none yet)

/home/upen/AEI8_New/11_Cert_Challenge/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/home/upen/AEI8_New/11_Cert_Challenge/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/home/upen/AEI8_New/11_Cert_Challenge/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


In [3]:
os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")
os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")
os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key:")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = f"AIM - RAGAS EVALS - {uuid4().hex[0:8]}"


In [4]:
BASELINE_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

In [5]:
from pathlib import Path

project_root = Path.cwd().parent  # Go up one level from notebooks/ to project root
data_path = project_root / "data/raw"

print(f"Project root: {project_root}")
print(f"Data path: {data_path}")
print(f"Data path exists: {data_path.exists()}")

Project root: /home/upen/AEI8_New/11_Cert_Challenge
Data path: /home/upen/AEI8_New/11_Cert_Challenge/data/raw
Data path exists: True


In [40]:
# Function to unzip files in data/raw directory
def unzip_data_files(data_dir: Path) -> None:
    """Unzip all zip files found in data directory"""
    import zipfile
    
    # Find all zip files
    zip_files = list(data_dir.glob("*.zip"))
    print(f"Found {len(zip_files)} zip files")
    
    # Extract each zip file
    for zip_file in zip_files:
        print(f"Extracting {zip_file.name}...")
        try:
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                # Extract to a directory named same as zip file without .zip extension
                extract_dir = zip_file.parent / zip_file.stem
                zip_ref.extractall(extract_dir)
            print(f"✓ Successfully extracted to {extract_dir}")
        except Exception as e:
            print(f"✗ Failed to extract {zip_file.name}: {str(e)}")

# Unzip any zip files in data directory
unzip_data_files(data_path)



Found 5 zip files
Extracting 20250828_b216af61-99d0-42a3-afc2-b5bf86df1ec0.zip...
✓ Successfully extracted to /home/upen/AEI8_New/11_Cert_Challenge/data/raw/20250828_b216af61-99d0-42a3-afc2-b5bf86df1ec0
Extracting 20250828_c18799cd-9e11-4196-9ab1-0a2a84585e56.zip...
✓ Successfully extracted to /home/upen/AEI8_New/11_Cert_Challenge/data/raw/20250828_c18799cd-9e11-4196-9ab1-0a2a84585e56
Extracting 20250828_cd60ded0-01d9-4a51-a8ad-cdfeb27d88fb.zip...
✓ Successfully extracted to /home/upen/AEI8_New/11_Cert_Challenge/data/raw/20250828_cd60ded0-01d9-4a51-a8ad-cdfeb27d88fb
Extracting 20250828_b40c9a90-7e2e-48d6-adb6-2351c0ca621d.zip...
✓ Successfully extracted to /home/upen/AEI8_New/11_Cert_Challenge/data/raw/20250828_b40c9a90-7e2e-48d6-adb6-2351c0ca621d
Extracting 20250827_f2318509-9a49-4c99-a08b-22381f17aacd.zip...
✓ Successfully extracted to /home/upen/AEI8_New/11_Cert_Challenge/data/raw/20250827_f2318509-9a49-4c99-a08b-22381f17aacd


In [41]:
# Enhanced function to find all XML files recursively
def find_xml_files(data_dir: Path) -> List[Path]:
    """Find all XML files in data directory and its subdirectories"""
    xml_files = []
    for xml_file in data_dir.rglob("*.xml"):
        xml_files.append(xml_file)
    return xml_files

# Get list of XML files
xml_files = find_xml_files(data_path)
print(f"Found {len(xml_files)} XML files")

# Print first few files to verify
print("\nFirst few XML files found:")
for xml_file in xml_files[:3]:
    print(f"- {xml_file.relative_to(data_path)}")

# Enhanced XML processing with better metadata extraction
from bs4 import BeautifulSoup
import pandas as pd
import re

def extract_drug_info_from_xml(xml_file: Path) -> dict:
    """Extract drug name and other metadata from XML file"""
    try:
        with open(xml_file) as f:
            soup = BeautifulSoup(f, 'xml')
        
        # Try to extract drug name from various possible locations
        drug_name = "Unknown Drug"
        
        # Look for drug name in title or name tags
        title_tag = soup.find('title')
        if title_tag and title_tag.text:
            drug_name = title_tag.text.strip()
        
        # Look for drug name in structured data
        name_tag = soup.find('name')
        if name_tag and name_tag.text:
            drug_name = name_tag.text.strip()
        
        # Extract section information if available
        section = "General"
        section_tag = soup.find('section')
        if section_tag and section_tag.text:
            section = section_tag.text.strip()
        
        return {
            'drug_name': drug_name,
            'section': section,
            'file_path': str(xml_file.relative_to(data_path))
        }
    except Exception as e:
        print(f"Error extracting metadata from {xml_file}: {e}")
        return {
            'drug_name': 'Unknown Drug',
            'section': 'General',
            'file_path': str(xml_file.relative_to(data_path))
        }

def extract_text_from_xml(xml_file: Path) -> List[dict]:
    """Extract text content from XML file with enhanced metadata and longer documents"""
    with open(xml_file) as f:
        soup = BeautifulSoup(f, 'xml')
    
    # Extract metadata
    metadata = extract_drug_info_from_xml(xml_file)
    
    # Extract text from paragraph tags and combine into longer documents
    paragraphs = []
    current_text = ""
    min_length = 200  # Minimum characters for a document
    
    for p in soup.find_all('paragraph'):
        if p.text.strip():
            text = p.text.strip()
            # Only include paragraphs with substantial content
            if len(text) > 50:  # Filter out very short paragraphs
                current_text += text + " "
                
                # If we have enough content, create a document
                if len(current_text) > min_length:
                    paragraphs.append({
                        'text': current_text.strip(),
                        'source': metadata['file_path'],
                        'drug_name': metadata['drug_name'],
                        'section': metadata['section']
                    })
                    current_text = ""  # Reset for next document
    
    # Add any remaining text if it's long enough
    if len(current_text.strip()) > min_length:
        paragraphs.append({
            'text': current_text.strip(),
            'source': metadata['file_path'],
            'drug_name': metadata['drug_name'],
            'section': metadata['section']
        })
    
    return paragraphs

# Process all XML files with enhanced metadata
all_paragraphs = []
for xml_file in xml_files:
    try:
        paragraphs = extract_text_from_xml(xml_file)
        all_paragraphs.extend(paragraphs)
    except Exception as e:
        print(f"Error processing {xml_file}: {e}")
        
# Convert to DataFrame
df = pd.DataFrame(all_paragraphs)

# Filter out documents that are too short for RAGAS
min_token_length = 100  # Minimum tokens for RAGAS
df = df[df['text'].str.len() > min_token_length]

print(f"\nExtracted {len(df)} documents total (after filtering short documents)")
print(f"Unique drugs found: {df['drug_name'].nunique()}")
print(f"Drugs: {df['drug_name'].unique()[:5]}")  # Show first 5 drugs
print("\nFirst few documents:")
print(df.head())

# Create datasets for different purposes
ragas_docs = df
retriever_docs = df

# Create a summary of the data
print(f"\nData Summary:")
print(f"- Total documents: {len(df)}")
print(f"- Unique drugs: {df['drug_name'].nunique()}")
print(f"- Unique sections: {df['section'].nunique()}")
print(f"- Average text length: {df['text'].str.len().mean():.1f} characters")
print(f"- Min text length: {df['text'].str.len().min()} characters")
print(f"- Max text length: {df['text'].str.len().max()} characters")

# Check if we have enough documents for RAGAS
if len(df) < 10:
    print(f"\n⚠️  Warning: Only {len(df)} documents available. RAGAS needs at least 10 documents.")
    print("Consider reducing the minimum length filter or processing more XML files.")
else:
    print(f"\n✅ Sufficient documents ({len(df)}) for RAGAS testset generation.")



Found 5 XML files

First few XML files found:
- 20250828_b216af61-99d0-42a3-afc2-b5bf86df1ec0/7325231c-0383-43aa-a3da-0b4580472456.xml
- 20250828_c18799cd-9e11-4196-9ab1-0a2a84585e56/e0683309-89ba-4804-887f-12fcaf59a2ca.xml
- 20250828_cd60ded0-01d9-4a51-a8ad-cdfeb27d88fb/334cb4f5-b33c-4871-9777-7bd9d0f4cab0.xml

Extracted 307 documents total (after filtering short documents)
Unique drugs found: 3
Drugs: ['Genentech, Inc.' 'Actavis Pharma, Inc.' 'Bryant Ranch Prepack']

First few documents:
                                                text  \
0  OCREVUS ZUNOVO is a CD20-directed cytolytic an...   
1  OCREVUS ZUNOVO should be administered via subc...   
2  Prior to initiating ocrelizumab treatment, per...   
3  Because vaccination with live-attenuated or li...   
4  Prior to initiating OCREVUS ZUNOVO, obtain ser...   

                                              source        drug_name  \
0  20250828_b216af61-99d0-42a3-afc2-b5bf86df1ec0/...  Genentech, Inc.   
1  20250828_b216af61-9

In [42]:
# RAGAS
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

# langchain_openai
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4.1-nano")

In [43]:
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)


In [44]:
# Convert DataFrame to LangChain Document objects with enhanced metadata
from langchain.schema import Document

# Convert DataFrame rows to Document objects with all metadata
ragas_documents = []
for _, row in ragas_docs.iterrows():
    doc = Document(
        page_content=row['text'],
        metadata={
            'source': row['source'],
            'drug_name': row['drug_name'],
            'section': row['section']
        }
    )
    ragas_documents.append(doc)

print(f"Created {len(ragas_documents)} LangChain Document objects")
print(f"Sample metadata: {ragas_documents[0].metadata}")

# Determine appropriate testset size based on available documents
available_docs = len(ragas_documents)
testset_size = min(10, available_docs // 2)  # Use half of available docs, max 10

print(f"\nGenerating testset with {testset_size} samples from {available_docs} documents...")

# Now generate testset with proper Document objects
try:
    golden_testset = generator.generate_with_langchain_docs(ragas_documents, testset_size=testset_size)
    print("✅ Testset generation successful!")
    print(f"Generated {len(golden_testset)} test samples")
    golden_testset.to_pandas()
except Exception as e:
    print(f"❌ Error generating testset: {e}")
    print("Try reducing testset_size or increasing document length requirements.")


Created 307 LangChain Document objects
Sample metadata: {'source': '20250828_b216af61-99d0-42a3-afc2-b5bf86df1ec0/7325231c-0383-43aa-a3da-0b4580472456.xml', 'drug_name': 'Genentech, Inc.', 'section': 'Ocrevus Zunovo\n\n\n\nocrelizumab and hyaluronidase\n\n\n\n\n\n\n\n\n\nOCRELIZUMAB\n\n\n\nOCRELIZUMAB\n\n\n\n\n\n\n\n\n\n\n\nHYALURONIDASE (HUMAN RECOMBINANT)\n\n\n\nHYALURONIDASE (HUMAN RECOMBINANT)\n\n\n\n\n\n\n\n\n\n\n\nTREHALOSE DIHYDRATE\n\n\n\n\n\n\n\n\n\nACETIC ACID\n\n\n\n\n\n\n\n\n\nMETHIONINE\n\n\n\n\n\n\n\n\n\nPOLYSORBATE 20\n\n\n\n\n\n\n\n\n\nSODIUM ACETATE\n\n\n\n\n\nWATER'}

Generating testset with 10 samples from 307 documents...


Applying SummaryExtractor:   0%|          | 0/126 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/307 [00:00<?, ?it/s]

Node 9c933173-952d-4f21-a9f2-501eb6ca146d does not have a summary. Skipping filtering.
Node af9064e7-e9bf-4f7a-acbb-25a0ffe5976b does not have a summary. Skipping filtering.
Node b6b910f9-0fb1-420a-bf78-ab5d018ce5bd does not have a summary. Skipping filtering.
Node 073ea5bb-6cea-4cd1-af46-5927e9daf1e5 does not have a summary. Skipping filtering.
Node 5d0d5bca-8962-4a23-83fb-885cf34aa212 does not have a summary. Skipping filtering.
Node 1736c897-d786-4aed-930b-966456f6dc9d does not have a summary. Skipping filtering.
Node bc425672-34a0-43d3-a104-228c470c5614 does not have a summary. Skipping filtering.
Node 8a919f7e-bc71-4f7f-b849-2249e26c084d does not have a summary. Skipping filtering.
Node d1f8641e-8216-45cf-a871-d423455bb31e does not have a summary. Skipping filtering.
Node bb3d8ccf-e36c-499f-a44a-5771ddd6b948 does not have a summary. Skipping filtering.
Node eb8db72b-4a3c-46bc-8187-33606c485243 does not have a summary. Skipping filtering.
Node ef2da35a-0b1f-413f-8cf9-93475f32eb29 d

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/740 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Testset generation successful!
Generated 10 test samples


In [45]:
golden_testset.to_pandas()



,user_input,reference_contexts,reference,synthesizer_name
0,Wher can OCREVUS ZUNOVO be injectd in the abod...,[OCREVUS ZUNOVO is a CD20-directed cytolytic a...,OCREVUS ZUNOVO is for subcutaneous use in the ...,single_hop_specifc_query_synthesizer
1,"How OCREVUS ZUNOVO use in old people, what nee...",[OCREVUS ZUNOVO should be administered via sub...,OCREVUS ZUNOVO should be given by subcutaneous...,single_hop_specifc_query_synthesizer
2,Why talk immunology before ocrelizumab?,"[Prior to initiating ocrelizumab treatment, pe...","For patients with low serum immunoglobulins, c...",single_hop_specifc_query_synthesizer
3,As a Neurology Clinical Research Coordinator o...,[Because vaccination with live-attenuated or l...,All immunizations should be administered accor...,single_hop_specifc_query_synthesizer
4,What is the recommended protocol regarding bil...,"[Prior to initiating OCREVUS ZUNOVO, obtain se...","Prior to initiating OCREVUS ZUNOVO, obtain ser...",single_hop_specifc_query_synthesizer
5,What are the key differences in pharmacokineti...,[<1-hop>\n\nPrior to treating patients with de...,Dextroamphetamine sulfate tablets (immediate-r...,multi_hop_specific_query_synthesizer
6,"In the context of Study 4, which compared the ...",[<1-hop>\n\nStudy 4 enrolled 236 patients (213...,"In Study 4, patients were divided into two gro...",multi_hop_specific_query_synthesizer
7,What considerations should be made regarding p...,[<1-hop>\n\nDextroamphetamine sulfate may prod...,When initiating treatment with dextroamphetami...,multi_hop_specific_query_synthesizer
8,What are the risks and necessary precautions w...,[<1-hop>\n\nIsosorbide Mononitrate Extended-Re...,When isosorbide mononitrate is used in elderly...,multi_hop_specific_query_synthesizer
9,what happen if old person take too much isosor...,[<1-hop>\n\nHemodynamic Effects\n\n The ill ef...,if old person take too much isosorbide mononit...,multi_hop_specific_query_synthesizer


In [46]:
baseline_dataset = copy.deepcopy(golden_testset)
rerank_dataset = copy.deepcopy(golden_testset)


In [ ]:
# Convert DataFrame rows to Document objects with all metadata and pass it to split_documents

In [49]:
# Convert DataFrame rows to Document objects
documents = []
for _, row in df.iterrows():
    doc = Document(
        page_content=row['text'],
        metadata={
            'source': row['source'],
            'drug_name': row['drug_name'], 
            'section': row['section']
        }
    )
    documents.append(doc)

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(documents)
print(f"Created {len(split_documents)} document chunks")


Created 474 document chunks


In [50]:
rag_prompt = ChatPromptTemplate.from_template(BASELINE_PROMPT)


In [53]:
baseline_client = QdrantClient(":memory:")

baseline_client.create_collection(
    collection_name="drug_labels",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

baseline_vector_store = QdrantVectorStore(
    client=baseline_client,
    collection_name="drug_labels",
    embedding=embeddings,
)


In [54]:
_ = baseline_vector_store.add_documents(documents=split_documents)

retriever = baseline_vector_store.as_retriever(search_kwargs={"k": 3})


In [55]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}


In [56]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

In [57]:
class State(TypedDict):
  question: str
  context: List[Document]
  response: str


In [58]:
baseline_graph_builder = StateGraph(State).add_sequence([retrieve, generate])
baseline_graph_builder.add_edge(START, "retrieve")
baseline_graph = baseline_graph_builder.compile()


In [73]:
retriever_docs = df

In [74]:
retriever_docs

,text,source,drug_name,section
0,OCREVUS ZUNOVO is a CD20-directed cytolytic an...,20250828_b216af61-99d0-42a3-afc2-b5bf86df1ec0/...,"Genentech, Inc.",Ocrevus Zunovo\n\n\n\nocrelizumab and hyaluron...
1,OCREVUS ZUNOVO should be administered via subc...,20250828_b216af61-99d0-42a3-afc2-b5bf86df1ec0/...,"Genentech, Inc.",Ocrevus Zunovo\n\n\n\nocrelizumab and hyaluron...
2,"Prior to initiating ocrelizumab treatment, per...",20250828_b216af61-99d0-42a3-afc2-b5bf86df1ec0/...,"Genentech, Inc.",Ocrevus Zunovo\n\n\n\nocrelizumab and hyaluron...
3,Because vaccination with live-attenuated or li...,20250828_b216af61-99d0-42a3-afc2-b5bf86df1ec0/...,"Genentech, Inc.",Ocrevus Zunovo\n\n\n\nocrelizumab and hyaluron...
4,"Prior to initiating OCREVUS ZUNOVO, obtain ser...",20250828_b216af61-99d0-42a3-afc2-b5bf86df1ec0/...,"Genentech, Inc.",Ocrevus Zunovo\n\n\n\nocrelizumab and hyaluron...
...,...,...,...,...
302,How should I store metformin hydrochloride ext...,20250827_f2318509-9a49-4c99-a08b-22381f17aacd/...,Bryant Ranch Prepack,METFORMIN HYDROCHLORIDE\n\n\n\nmetformin hydro...
303,General information about the use of metformin...,20250827_f2318509-9a49-4c99-a08b-22381f17aacd/...,Bryant Ranch Prepack,METFORMIN HYDROCHLORIDE\n\n\n\nmetformin hydro...
304,"For more information, call Ascend Laboratories...",20250827_f2318509-9a49-4c99-a08b-22381f17aacd/...,Bryant Ranch Prepack,METFORMIN HYDROCHLORIDE\n\n\n\nmetformin hydro...
305,Inactive ingredients in each tablet of metform...,20250827_f2318509-9a49-4c99-a08b-22381f17aacd/...,Bryant Ranch Prepack,METFORMIN HYDROCHLORIDE\n\n\n\nmetformin hydro...


In [75]:
#Convert DataFrame rows to Document objects with all metadata
retriever_documents= []
for _, row in retriever_docs.iterrows():
    doc = Document(
        page_content=row['text'],
        metadata={
            'source': row['source'],
            'drug_name': row['drug_name'],
            'section': row['section']
        }
    )
    retriever_documents.append(doc)

print(f"Created {len(retriever_documents)} LangChain Document objects")
print(f"Sample metadata: {retriever_documents[0].metadata}")

Created 307 LangChain Document objects
Sample metadata: {'source': '20250828_b216af61-99d0-42a3-afc2-b5bf86df1ec0/7325231c-0383-43aa-a3da-0b4580472456.xml', 'drug_name': 'Genentech, Inc.', 'section': 'Ocrevus Zunovo\n\n\n\nocrelizumab and hyaluronidase\n\n\n\n\n\n\n\n\n\nOCRELIZUMAB\n\n\n\nOCRELIZUMAB\n\n\n\n\n\n\n\n\n\n\n\nHYALURONIDASE (HUMAN RECOMBINANT)\n\n\n\nHYALURONIDASE (HUMAN RECOMBINANT)\n\n\n\n\n\n\n\n\n\n\n\nTREHALOSE DIHYDRATE\n\n\n\n\n\n\n\n\n\nACETIC ACID\n\n\n\n\n\n\n\n\n\nMETHIONINE\n\n\n\n\n\n\n\n\n\nPOLYSORBATE 20\n\n\n\n\n\n\n\n\n\nSODIUM ACETATE\n\n\n\n\n\nWATER'}


In [76]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(retriever_documents)
len(split_documents)


474

In [77]:
rerank_client = QdrantClient(":memory:")

rerank_client.create_collection(
    collection_name="use_case_data_new_chunks",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

rerank_vector_store = QdrantVectorStore(
    client=rerank_client,
    collection_name="use_case_data_new_chunks",
    embedding=embeddings,
)


In [78]:
_ = rerank_vector_store.add_documents(documents=split_documents)

baseline_retriever = rerank_vector_store.as_retriever(search_kwargs={"k": 20})


In [79]:
def retrieve_reranked(state):
  compressor = CohereRerank(model="rerank-v3.5")
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=baseline_retriever, search_kwargs={"k": 5}
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context" : retrieved_docs}


In [80]:
class AdjustedState(TypedDict):
  question: str
  context: List[Document]
  response: str

rerank_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_reranked, generate])
rerank_graph_builder.add_edge(START, "retrieve_reranked")
rerank_graph = rerank_graph_builder.compile()


In [81]:
custom_run_config = RunConfig(timeout=360)


In [82]:
for test_row in baseline_dataset:
  response = baseline_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]


In [83]:
baseline_evaluation_dataset = EvaluationDataset.from_pandas(baseline_dataset.to_pandas())


In [84]:
baseline_evaluation_dataset.to_csv("baseline_evaluation_dataset.csv")


In [85]:
for test_row in rerank_dataset:
  response = rerank_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(2) # To try to avoid rate limiting.


In [86]:
rerank_evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())


In [87]:
rerank_evaluation_dataset.to_csv("rerank_evaluation_dataset.csv")


In [88]:
baseline_result = evaluate(
    dataset=baseline_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Task exception was never retrieved
future: <Task finished name='Task-8100' coro=<as_completed.<locals>.sema_coro() done, defined at /home/upen/AEI8_New/11_Cert_Challenge/.venv/lib/python3.13/site-packages/ragas/executor.py:46> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/home/upen/AEI8_New/11_Cert_Challenge/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_20757/2165245558.py", line 28, in <module>
    golden_testset = generator.generate_with_langchain_docs(ragas_documents, testset_size=testset_size)
  File "/home/upen/AEI8_New/11_Cert_Challenge/.venv/lib/python3.13/site-packages/ragas/testset/synthesizers/generate.py", line 185, in generate_with_langchain_docs
    apply_transforms(kg, transforms)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/home/upen/AEI8_New/11_Cert_Challenge/.venv/

In [89]:
rerank_evaluation_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

In [90]:
baseline_result


{'context_recall': 0.6200, 'faithfulness': 0.6742, 'factual_correctness': 0.5000, 'answer_relevancy': 0.8550, 'context_entity_recall': 0.4560, 'noise_sensitivity_relevant': 0.0933}

In [91]:
rerank_evaluation_result


{'context_recall': 0.8033, 'faithfulness': 0.7704, 'factual_correctness': 0.6850, 'answer_relevancy': 0.9491, 'context_entity_recall': 0.6861, 'noise_sensitivity_relevant': 0.1252}

In [96]:
import pandas as pd
import numpy as np

# Create a DataFrame for comparison
metrics = ['context_recall', 'faithfulness', 'factual_correctness', 'answer_relevancy', 'context_entity_recall', 'noise_sensitivity_relevant']

# Convert results to float values by taking mean if they are lists
def get_metric_value(result, metric):
    value = result[metric]
    return np.mean(value) if isinstance(value, list) else value

comparison_df = pd.DataFrame({
    'Metric': metrics,
    'Baseline': [get_metric_value(baseline_result, metric) for metric in metrics],
    'Reranked': [get_metric_value(rerank_evaluation_result, metric) for metric in metrics],
})

# Calculate difference after ensuring values are floats
comparison_df['Difference'] = comparison_df['Reranked'] - comparison_df['Baseline']

# Round all numeric columns
comparison_df = comparison_df.round(4)

# Display the comparison with better formatting
pd.set_option('display.float_format', '{:.4f}'.format)
comparison_df





,Metric,Baseline,Reranked,Difference
0,context_recall,0.6200,0.8033,0.1833
1,faithfulness,0.6742,0.7704,0.0962
2,factual_correctness,0.5000,0.6850,0.1850
3,answer_relevancy,0.8550,0.9491,0.0940
4,context_entity_recall,0.4560,0.6861,0.2300
5,noise_sensitivity_relevant,0.0933,0.1252,0.0319
